In [1]:
import os
import glob
import tiktoken
import numpy as np
import fitz  
from dotenv import load_dotenv
import base64
import requests
import time
import edge_tts
import asyncio
import uuid
import threading


from langchain_openai import OpenAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyMuPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document


from groq import Groq 
import gradio as gr   
from gtts import gTTS 


from sklearn.manifold import TSNE
import plotly.graph_objects as go

In [2]:

TEXT_MODEL = "llama-3.1-8b-instant"

# Image/PDF Vision සඳහා Model එක
#VISION_MODEL = "llama-3.2-11b-vision-instant"

In [3]:
load_dotenv(override=True)


api_key = os.getenv('GROQ_API_KEY')
#hf_token = os.getenv("HF_TOKEN")


client = Groq(api_key=api_key)
#client = OpenAI(api_key=api_key, base_url="https://api.groq.com/openai/v1")


db_name = "vector_db"
persist_folder = "my_vector_db"

In [ ]:
###### DO NOT RUN AGAIN

knowledge_base_path = "knowledge-base/**/*.pdf"
files = glob.glob(knowledge_base_path, recursive=True)
print(f"Found {len(files)} PDF files")

all_documents = []


for file_path in files:
    print(f"Loading: {file_path}")
    loader = PyMuPDFLoader(file_path)
    
    all_documents.extend(loader.load())


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=100
)
chunks = text_splitter.split_documents(all_documents)

print(f"සම්පූර්ණ ලේඛන කැබලි (Chunks) ගණන: {len(chunks)}")

Found 3 PDF files
Loading: knowledge-base\(by-John-Paul-Mueller,-Luca-Massaron)-Artificial-I.pdf
Loading: knowledge-base\Grokking Deep Learning by Andrew W. Trask (z-lib.org).pdf
Loading: knowledge-base\tensorflow-2-pocket-primer@NetworkArtificial.pdf
සම්පූර්ණ ලේඛන කැබලි (Chunks) ගණන: 2408


In [ ]:
###### DO NOT RUN AGAIN

print("Embedding model එක load වෙනවා...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


print(f"Vector Database එක නිර්මාණය වෙනවා ({persist_folder})... මේකට පොඩ්ඩක් වෙලා යයි.")

vector_db = Chroma.from_documents(
    documents=chunks, 
    embedding=embeddings, 
    persist_directory=persist_folder
)

print("වැඩේ ගොඩ! ඔයාගේ පොත් වල දත්ත දැන් Vector Database එකක් විදියට save වෙලා තියෙන්නේ.")

Embedding model එක load වෙනවා...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector Database එක නිර්මාණය වෙනවා (my_vector_db)... මේකට පොඩ්ඩක් වෙලා යයි.
වැඩේ ගොඩ! ඔයාගේ පොත් වල දත්ත දැන් Vector Database එකක් විදියට save වෙලා තියෙන්නේ.


In [ ]:
###### DO NOT RUN AGAIN

HF_TOKEN = os.getenv("HF_TOKEN")

API_URL = "https://api-inference.huggingface.co/models/microsoft/Florence-2-large"
headers = {"Authorization": f"Bearer {HF_TOKEN}"}

def describe_image_with_hf(image_bytes):
    """Groq වෙනුවට Hugging Face හරහා image එක describe කරන function එක"""
    try:
        
        response = requests.post(API_URL, headers=headers, data=image_bytes)
        
        if response.status_code == 200:
            result = response.json()
            
            if isinstance(result, list) and len(result) > 0:
                return result[0].get('generated_text', 'No description generated.')
            return str(result)
        else:
            return f"Error: API status code {response.status_code}"
            
    except Exception as e:
        return f"Hugging Face processing error: {str(e)}"


def extract_and_describe_images(pdf_path):
    doc = fitz.open(pdf_path)
    image_descriptions = []
    
    for page_index in range(len(doc)):
        page = doc[page_index]
        image_list = page.get_images(full=True)
        
        for img_index, img in enumerate(image_list):
            xref = img[0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]
            
            
            print(f"Describing image {img_index+1} on page {page_index+1}...")
            description = describe_image_with_hf(image_bytes)
            
            image_descriptions.append(f"Image on page {page_index+1}: {description}")
            
    return image_descriptions

In [ ]:
###### DO NOT RUN AGAIN


pdf_files = glob.glob("knowledge-base/**/*.pdf", recursive=True)

print("Images process කරනවා... මේකට Hugging Face Inference API (Florence-2) පාවිච්චි වෙනවා.")

image_chunks = []

for pdf in pdf_files:
    print(f"පොත පරීක්ෂා කරනවා: {pdf}")
    
    descriptions = extract_and_describe_images(pdf)
    
    for desc in descriptions:
        
        doc = Document(
            page_content=desc, 
            metadata={
                "source": pdf, 
                "type": "image_description",
                "method": "huggingface_florence2"
            }
        )
        image_chunks.append(doc)
    
    
    time.sleep(2)


if image_chunks:
    print(f"Database එකට chunks {len(image_chunks)}ක් එකතු කරනවා...")
    vector_db.add_documents(image_chunks)
    
    print(f"සාර්ථකයි! Images {len(image_chunks)} ක විස්තර Database එකට එකතු කළා.")
else:
    print("කිසිදු Image එකක් හමු වුණේ නැහැ මචං.")

Images process කරනවා... මේකට Hugging Face Inference API (Florence-2) පාවිච්චි වෙනවා.
පොත පරීක්ෂා කරනවා: knowledge-base\(by-John-Paul-Mueller,-Luca-Massaron)-Artificial-I.pdf
Describing image 1 on page 1...
Describing image 1 on page 39...
Describing image 1 on page 58...
Describing image 1 on page 59...
Describing image 1 on page 61...
Describing image 1 on page 128...
Describing image 1 on page 174...
Describing image 1 on page 192...
Describing image 2 on page 192...
Describing image 3 on page 192...
Describing image 4 on page 192...
Describing image 5 on page 192...
Describing image 6 on page 192...
Describing image 7 on page 192...
Describing image 1 on page 215...
Describing image 2 on page 215...
Describing image 3 on page 215...
Describing image 4 on page 215...
Describing image 5 on page 215...
Describing image 6 on page 215...
Describing image 7 on page 215...
Describing image 8 on page 215...
Describing image 9 on page 215...
Describing image 10 on page 215...
Describing imag

In [4]:

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


vector_db = Chroma(
    persist_directory=persist_folder, 
    embedding_function=embeddings
)

print("✅ Database Loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Database Loaded successfully!


In [5]:
import plotly.express as px

def visualize_embeddings_enhanced():
    print("Fetching data from Vector DB...")
    
    data = vector_db.get(include=['embeddings', 'metadatas', 'documents'])
    
    if len(data['embeddings']) == 0:
        print("කනගාටුයි මචං, DB එකේ Embeddings මොකුත් නැහැ.")
        return

    embeddings = np.array(data['embeddings'])
    sources = [os.path.basename(m.get('source', 'Unknown')) for m in data['metadatas']]
    content_types = [m.get('type', 'text') for m in data['metadatas']]
    texts = [d[:100] + "..." for d in data['documents']]

    
    symbols = {'text': 'circle', 'image_description': 'diamond'}
    
    # --- 2D Visualization ---
    print("Generating 2D Visualization (Optimized)...")
    
    tsne_2d = TSNE(n_components=2, perplexity=40, max_iter=1000, random_state=42)
    embeddings_2d = tsne_2d.fit_transform(embeddings)

    fig_2d = go.Figure()
    for source in set(sources):
        for c_type in set(content_types):
            indices = [i for i, (s, t) in enumerate(zip(sources, content_types)) if s == source and t == c_type]
            if not indices: continue
            
            fig_2d.add_trace(go.Scatter(
                x=embeddings_2d[indices, 0], y=embeddings_2d[indices, 1],
                mode='markers',
                name=f"{source} ({c_type})",
                text=[texts[i] for i in indices],
                marker=dict(
                    size=9, 
                    symbol=symbols.get(c_type, 'circle'),
                    opacity=0.8,
                    line=dict(width=1, color='white') # තිත වටේ සුදු පාට ඉරක් (Clear පේන්න)
                )
            ))
    
    fig_2d.update_layout(
        title="<b>2D Vector Space: Enhanced Clustering</b>",
        template="plotly_dark",
        legend_title="Sources & Types",
        margin=dict(l=0, r=0, b=0, t=50)
    )
    fig_2d.show()

    # --- 3D Visualization ---
    print("Generating 3D Visualization (Optimized)...")
    tsne_3d = TSNE(n_components=3, perplexity=40, max_iter=1000, random_state=42)
    embeddings_3d = tsne_3d.fit_transform(embeddings)

    fig_3d = go.Figure()
    for source in set(sources):
        for c_type in set(content_types):
            indices = [i for i, (s, t) in enumerate(zip(sources, content_types)) if s == source and t == c_type]
            if not indices: continue

            fig_3d.add_trace(go.Scatter3d(
                x=embeddings_3d[indices, 0], y=embeddings_3d[indices, 1], z=embeddings_3d[indices, 2],
                mode='markers',
                name=f"{source} ({c_type})",
                text=[texts[i] for i in indices],
                marker=dict(
                    size=5, 
                    symbol=symbols.get(c_type, 'circle'),
                    opacity=0.8,
                    line=dict(width=0.5, color='white')
                )
            ))

    fig_3d.update_layout(
        title="<b>3D Vector Space: Enhanced Exploration</b>",
        template="plotly_dark",
        scene=dict(
            xaxis_title='TSNE 1',
            yaxis_title='TSNE 2',
            zaxis_title='TSNE 3'
        ),
        margin=dict(l=0, r=0, b=0, t=50)
    )
    fig_3d.show()


visualize_embeddings_enhanced()

Fetching data from Vector DB...
Generating 2D Visualization (Optimized)...


Generating 3D Visualization (Optimized)...


In [6]:

def delete_later(file_path, delay=3600): 
    def delayed_delete():
        time.sleep(delay)
        try:
            if os.path.exists(file_path):
                os.remove(file_path)
                
        except Exception as e:
            pass 
    
    
    threading.Thread(target=delayed_delete, daemon=True).start()


def chatbot_response(user_query):
    try:
        
        docs = vector_db.similarity_search(user_query, k=5)
        
        context_parts = []
        for d in docs:
            source = os.path.basename(d.metadata.get('source', 'Unknown'))
            content_type = d.metadata.get('type', 'text')
            context_parts.append(f"[{content_type} from {source}]: {d.page_content}")
            
        context = "\n\n".join(context_parts)
        
        system_prompt = f"""You are an expert AI Educator specialized in Deep Learning. Your goal is to explain complex concepts from the provided book context in a clear, conversational, and highly understandable manner.

        Follow these instructions strictly:
        1. **Analyze and Synthesize**: Do not just copy-paste text from the context. Read the provided excerpts and explain the underlying concept as if you are teaching a student.
        2. **Explain the 'Why' and 'How'**: When a concept or function (like tf.constant) is mentioned, explain why it is used and how it works according to the book's logic.
        3. **Incorporate Visuals**: If the context includes image descriptions (e.g., Figure X), refer to them naturally in your explanation (e.g., "As illustrated in the diagram of the neural network...").
        4. **Tone**: Maintain a professional yet accessible tone. Use analogies if they help clarify the book's points.
        5. **Grounding**: Ensure your explanation is rooted in the provided books. If you add general knowledge to improve clarity, make sure it does not contradict the book's content.
        6. **Structure**: Use bullet points and bold text to highlight key terms and make the answer easy to scan.

        Context from Books:
        {context}

        If the context does not contain enough information, state that clearly but provide a helpful explanation based on general AI principles."""
        
        
        chat_completion = client.chat.completions.create(
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_query}
            ],
            model="llama-3.1-8b-instant", 
        )
        
        answer = chat_completion.choices[0].message.content
        
        
        unique_id = str(uuid.uuid4())
        audio_file = f"speech_{unique_id}.mp3"
        
        communicate = edge_tts.Communicate(answer, "en-US-AndrewNeural")
        asyncio.run(communicate.save(audio_file))
        
        
        delete_later(audio_file, delay=3600)
        
        return answer, audio_file
        
    except Exception as e:
        return f"Error occurred: {str(e)}", None


custom_theme = gr.themes.Soft(
    primary_hue="indigo",
    secondary_hue="blue",
)

with gr.Blocks(theme=custom_theme) as demo:
    gr.Markdown("<h1 style='text-align: center;'>🧠 DeepScholar AI: Multi-Modal RAG</h1>")
    gr.Markdown("<p style='text-align: center; font-size: 16px;'>Explore AI concepts with visual and textual knowledge from premium deep learning books.</p>")
    
    with gr.Accordion("📚 Knowledge Base (Powered by RAG)", open=False):
        gr.Markdown("""
        **This AI is grounded on the following textbooks:**
        1. *Artificial Intelligence For Dummies* - John Paul Mueller & Luca Massaron
        2. *Grokking Deep Learning* - Andrew W. Trask
        3. *TensorFlow 2.0 Pocket Primer* - Oswald Campesato
        """)
    
    gr.Markdown("---")
    
    with gr.Row():
        with gr.Column(scale=1):
            query_input = gr.Textbox(
                label="Ask your question:", 
                placeholder="e.g., Explain the structure of a neural network...",
                lines=4
            )
            ask_btn = gr.Button("Generate Answer 🚀", variant="primary")
            
            gr.Markdown("<br>### 🎧 Audio Response")
            audio_output = gr.Audio(label="Listen to Answer", autoplay=True)
            
        with gr.Column(scale=2):
            text_output = gr.Textbox(
                label="AI Response", 
                placeholder="The generated answer will appear here...",
                interactive=False, 
                lines=18
            )

    ask_btn.click(
        fn=chatbot_response, 
        inputs=query_input, 
        outputs=[text_output, audio_output]
    )


demo.launch(share=True)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_21944\3278033875.py:75: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=custom_theme) as demo:


* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


Exception in callback _ProactorBasePipeTransport._call_connection_lost()
handle: <Handle _ProactorBasePipeTransport._call_connection_lost()>
Traceback (most recent call last):
  File "C:\Users\ASUS\AppData\Local\Programs\Python\Python314\Lib\asyncio\events.py", line 94, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\ASUS\AppData\Local\Programs\Python\Python314\Lib\asyncio\proactor_events.py", line 165, in _call_connection_lost
    self._sock.shutdown(socket.SHUT_RDWR)
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^
ConnectionResetError: [WinError 10054] An existing connection was forcibly closed by the remote host
